# 🚀 Ollama Model Quantizer

Quantizza modelli Hugging Face in formato GGUF per Ollama.

Basato su:
* [Ollama](https://ollama.com/) - Framework per modelli locali
* [GGUF](https://github.com/ggerganov/ggml) - Formato di quantizzazione
* [Hugging Face](https://huggingface.co/) - Repository modelli

### ⭕ Disclaimer
The purpose of this document is to research bleeding-edge technologies in the field of machine learning.
Please read and follow the [Google Colab guidelines](https://research.google.com/colaboratory/faq.html) and its [Terms of Service](https://research.google.com/colaboratory/tos_v3.html).

## 📦 Installation Cell

**This cell installs all necessary dependencies:**
* UV package manager
* Hugging Face Hub & Transformers
* Ollama framework
* Mounts Google Drive
* Starts Ollama service

**⏱️ Estimated time:** 2-3 minutes

Run the cell below to begin installation.

In [ ]:
#@title 📦 Run Installation

import os
import subprocess
import time
from pathlib import Path

def install_dependencies():
    print("\n📦 Installing dependencies...\n")
    t0 = time.time()

    # Installa UV
    if not os.path.exists(os.path.expanduser("~/.cargo/bin/uv")):
        subprocess.run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True)
        os.environ["PATH"] = f"{os.path.expanduser('~')}/.cargo/bin:" + os.environ.get("PATH", "")

    # Installa pacchetti Python
    packages = ["huggingface_hub", "transformers"]
    for pkg in packages:
        subprocess.run(f"uv pip install -q {pkg}", shell=True)

    # Installa Ollama
    if not os.path.exists("/usr/local/bin/ollama"):
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

    # Avvia servizio Ollama
    subprocess.Popen("ollama serve > /tmp/ollama.log 2>&1 &", shell=True)

    # Attendi che il servizio sia pronto
    for i in range(10):
        time.sleep(2)
        result = subprocess.run("curl -s http://localhost:11434/api/tags > /dev/null", shell=True)
        if result.returncode == 0:
            break

    t1 = time.time()
    print(f"\n✅ Installation completed in {int(t1-t0)} seconds.")

# Monta Google Drive se necessario
if not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

# Installa dipendenze
install_dependencies()

## 📊 Quantization Guide

GGUF (GGUF Universal Format) è il formato nativo di llama.cpp e Ollama. La quantizzazione riduce la dimensione del modello sacrificando un minimo di qualità.

### 📈 Quality vs Size Trade-off

| Tipo | Bits | Qualità | Dimensione Relativa | Utilizzo Consigliato |
|------|------|---------|-------------------|---------------------|
| **fp32** | 32 | ★★★★★ | 100% | CPU pura, massima precisione |
| **fp16** | 16 | ★★★★★ | 50% | GPU moderne, qualità lossless |
| **q8_0** | 8.0 | ★★★★☆ | 25% | Quasi lossless, ottimo compromesso |
| **q6_K** | 6.5 | ★★★★☆ | 20% | Eccellente per la maggior parte dei casi |
| **q5_K_M** | 5.5 | ★★★★☆ | 17% | Qualità molto alta, buona compressione |
| **q5_K_S** | 5.5 | ★★★☆☆ | 17% | Qualità alta, leggero risparmio |
| **q4_K_M** | 4.5 | ★★★★☆ | 14% | 🏆 **MIGLIOR BILANCIAMENTO** qualità/dimensione |
| **q4_K_S** | 4.5 | ★★★☆☆ | 14% | Buon bilanciamento per uso generale |
| **iq4_xs** | 4.25 | ★★★☆☆ | 13% | Migliore qualità per dimensione con I-Matrix |
| **q3_K_M** | 3.5 | ★★★☆☆ | 11% | Buona compressione per dispositivi limitati |
| **q3_K_S** | 3.5 | ★★☆☆☆ | 11% | Alta compressione per dispositivi molto limitati |
| **iq3_m** | 3.66 | ★★☆☆☆ | 11% | Compressione media con I-Matrix |
| **iq3_xs** | 3.3 | ★★☆☆☆ | 10% | Alta compressione con I-Matrix |
| **q2_K** | 2.5 | ★☆☆☆☆ | 8% | Massima compressione, qualità minima |
| **iq2_m** | 2.4 | ★☆☆☆☆ | 7% | Compressione estrema con I-Matrix |
| **iq2_xs** | 2.2 | ★☆☆☆☆ | 7% | Compressione estrema per test |

### 🎯 Come scegliere:

1. **Per uso professionale/scientifico**: fp16 o q8_0
2. **Miglior compromesso qualità/dimensione**: q4_K_M (raccomandato)
3. **Dispositivi con RAM limitata (8-16GB)**: q4_K_M o q4_K_S
4. **Dispositivi molto limitati (4-8GB)**: q3_K_M o q3_K_S
5. **Solo per test/sperimentazione**: q2_K o versioni iq

### 📝 Nota sulle versioni:
- **K-means** (q*_K_*): Ottimizzazioni K-means per migliore qualità
- **I-Matrix** (iq*_*): Utilizza Importance Matrix per preservare i pesi più importanti
- **S/M/L**: Small, Medium, Large indicano diverse configurazioni di ottimizzazione

### 💡 Suggerimento:
Per la maggior parte degli utenti, **q4_K_M** offre il miglior bilanciamento tra qualità e dimensione. Puoi anche generare multiple quantizzazioni per testare quale funziona meglio sul tuo hardware.

# 🚀 Start Quantization

Configura il modello e avvia la quantizzazione.

In [ ]:
#@title 🌟 Configure & Quantize Model

# ============================================
# QUANTIZZAZIONE - TUTTO IN UNO
# ============================================

import os
import re
import subprocess
import time
import shutil
from pathlib import Path
from IPython.display import Markdown, display, clear_output
import warnings

warnings.filterwarnings("ignore")

# ============================================
# CONFIGURAZIONE UTENTE
# ============================================

#@markdown ## 🌐 Model Selection

#@markdown Incolla l'URL o l'ID del modello da Hugging Face
#@markdown Esempi:
#@markdown - `https://huggingface.co/meta-llama/Llama-2-7b-hf`
#@markdown - `mistralai/Mistral-7B-v0.1`
#@markdown - `google/gemma-2b-it`
model_url = "https://huggingface.co/meta-llama/Llama-2-7b-hf" #@param {type:"string"}

#@markdown ## ⚙️ Quantization Selection

#@markdown Scegli le quantizzazioni da generare (numeri separati da virgola)
#@markdown Esempio: `3,5,7` per q8_0, q5_K_M, q4_K_M
#@markdown Usa `all` per tutte le quantizzazioni
#@markdown
#@markdown **Legenda numeri:**
#@markdown - 1=fp32, 2=fp16, 3=q8_0, 4=q6_K, 5=q5_K_M, 6=q5_K_S, 7=q4_K_M (consigliato), 8=q4_K_S
#@markdown - 9=iq4_xs, 10=q3_K_M, 11=q3_K_S, 12=iq3_m, 13=iq3_xs, 14=q2_K, 15=iq2_m, 16=iq2_xs
quant_choice = "7" #@param {type:"string"}

#@markdown ## 📁 Output Options

#@markdown I modelli quantizzati verranno salvati in Google Drive
folder_structure = "Organize by project (MyDrive/quantized_models/project_name)" #@param ["Organize by project (MyDrive/quantized_models/project_name)", "Single folder (MyDrive/quantized_models)"]

#@markdown ## 🔧 Options

skip_existing = True #@param {type:"boolean"}
clear_output_between_quants = True #@param {type:"boolean"}

# ============================================
# QUANTIZZAZIONI DISPONIBILI
# ============================================

QUANTIZATIONS = {
    "1": {"name": "fp32", "bits": 32, "type": "float", "desc": "32-bit floating point (lossless, CPU only)", "quality": "★★★★★"},
    "2": {"name": "fp16", "bits": 16, "type": "float", "desc": "16-bit floating point (lossless, GPU ottimizzato)", "quality": "★★★★★"},
    "3": {"name": "q8_0", "bits": 8.0, "type": "int", "desc": "8-bit integer (quasi lossless)", "quality": "★★★★☆"},
    "4": {"name": "q6_K", "bits": 6.5, "type": "int", "desc": "6-bit K-means (quasi lossless)", "quality": "★★★★☆"},
    "5": {"name": "q5_K_M", "bits": 5.5, "type": "int", "desc": "5-bit K-means medium (qualità molto alta)", "quality": "★★★★☆"},
    "6": {"name": "q5_K_S", "bits": 5.5, "type": "int", "desc": "5-bit K-means small (qualità alta)", "quality": "★★★☆☆"},
    "7": {"name": "q4_K_M", "bits": 4.5, "type": "int", "desc": "4-bit K-means medium - 🏆 MIGLIOR BILANCIAMENTO", "quality": "★★★★☆"},
    "8": {"name": "q4_K_S", "bits": 4.5, "type": "int", "desc": "4-bit K-means small (buon bilanciamento)", "quality": "★★★☆☆"},
    "9": {"name": "iq4_xs", "bits": 4.25, "type": "int", "desc": "I-Matrix 4.25-bit (migliore size/qualità)", "quality": "★★★☆☆"},
    "10": {"name": "q3_K_M", "bits": 3.5, "type": "int", "desc": "3-bit K-means medium (buona compressione)", "quality": "★★★☆☆"},
    "11": {"name": "q3_K_S", "bits": 3.5, "type": "int", "desc": "3-bit K-means small (alta compressione)", "quality": "★★☆☆☆"},
    "12": {"name": "iq3_m", "bits": 3.66, "type": "int", "desc": "I-Matrix 3.66-bit (compressione media)", "quality": "★★☆☆☆"},
    "13": {"name": "iq3_xs", "bits": 3.3, "type": "int", "desc": "I-Matrix 3.3-bit (alta compressione)", "quality": "★★☆☆☆"},
    "14": {"name": "q2_K", "bits": 2.5, "type": "int", "desc": "2-bit K-means (massima compressione)", "quality": "★☆☆☆☆"},
    "15": {"name": "iq2_m", "bits": 2.4, "type": "int", "desc": "I-Matrix 2.4-bit (compressione estrema)", "quality": "★☆☆☆☆"},
    "16": {"name": "iq2_xs", "bits": 2.2, "type": "int", "desc": "I-Matrix 2.2-bit (compressione estrema)", "quality": "★☆☆☆☆"},
}

def parse_quant_selection(choice_str):
    """Parse user's quantization selection"""
    if choice_str.lower() == "all":
        return [QUANTIZATIONS[k]["name"] for k in QUANTIZATIONS]

    selected = []
    for part in choice_str.split(","):
        part = part.strip()
        if part in QUANTIZATIONS:
            selected.append(QUANTIZATIONS[part]["name"])
    return selected

# ============================================
# DOWNLOAD DEL MODELLO
# ============================================

def download_model():
    """Download model from Hugging Face"""
    print(f"\n📥 Downloading model from: {model_url}")

    # Estrai repo_id dall'URL
    if "huggingface.co/" in model_url:
        repo_id = model_url.split("huggingface.co/")[1]
    else:
        repo_id = model_url

    model_dir = f"/content/models/{repo_id.replace('/', '_')}"

    # Verifica se già scaricato
    if os.path.exists(model_dir) and any(os.listdir(model_dir)):
        print(f"✅ Model already exists at {model_dir}")
        return model_dir

    from huggingface_hub import snapshot_download

    try:
        print("⏳ Downloading... (this may take several minutes)")
        snapshot_download(
            repo_id=repo_id,
            local_dir=model_dir,
            local_dir_use_symlinks=False,
            resume_download=True,
            ignore_patterns=["*.safetensors.index.json", "*.msgpack", "*.h5"]
        )

        # Calcola dimensione
        total_size = 0
        for dirpath, dirnames, filenames in os.walk(model_dir):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                total_size += os.path.getsize(fp)

        size_gb = total_size / (1024**3)
        print(f"✅ Downloaded {size_gb:.2f} GB to {model_dir}")
        return model_dir
    except Exception as e:
        print(f"❌ Download failed: {e}")
        return None

# ============================================
# QUANTIZZAZIONE
# ============================================

def quantize_model(model_path, quant_name, model_name, output_dir):
    """Quantizza il modello usando Ollama"""

    tag = f"{model_name}:{quant_name}"
    output_file = os.path.join(output_dir, f"{model_name}-{quant_name}.gguf")

    # Salta se già esiste
    if skip_existing and os.path.exists(output_file):
        print(f"⏭️ Skipping {quant_name} - already exists")
        return True

    print(f"\n🔄 Quantizing {quant_name}...")
    start_time = time.time()

    # Crea Modelfile
    modelfile_content = f"FROM {model_path}\n"
    modelfile_path = f"/tmp/Modelfile_{quant_name}"

    with open(modelfile_path, 'w') as f:
        f.write(modelfile_content)

    # Esegui quantizzazione
    cmd = f"ollama create {tag} --quantize {quant_name} --file {modelfile_path}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    elapsed = time.time() - start_time

    if result.returncode == 0:
        print(f"✅ {quant_name} completed in {elapsed:.1f}s")

        # Cerca e copia il file GGUF
        ollama_models = Path.home() / ".ollama" / "models"
        gguf_files = list(ollama_models.rglob("*.gguf"))

        if gguf_files:
            latest_gguf = max(gguf_files, key=os.path.getctime)
            if os.path.getsize(latest_gguf) > 100 * 1024 * 1024:
                shutil.copy2(latest_gguf, output_file)
                file_size = os.path.getsize(output_file) / (1024**3)
                print(f"💾 Saved: {model_name}-{quant_name}.gguf ({file_size:.2f} GB)")

                # Rimuovi da Ollama per liberare spazio
                subprocess.run(f"ollama rm {tag}", shell=True, capture_output=True)
                return True

        # Fallback: cerca nei blob
        blob_dir = ollama_models / "blobs"
        if blob_dir.exists():
            recent_files = sorted(blob_dir.iterdir(), key=os.path.getctime, reverse=True)
            for f in recent_files[:5]:
                if f.stat().st_size > 100 * 1024 * 1024:
                    shutil.copy2(f, output_file)
                    print(f"💾 Saved from blob: {model_name}-{quant_name}.gguf")
                    subprocess.run(f"ollama rm {tag}", shell=True, capture_output=True)
                    return True

        print(f"⚠️ GGUF file not found, but model may be in Ollama")
        return True
    else:
        print(f"❌ {quant_name} failed: {result.stderr[:200]}")
        return False

# ============================================
# MAIN EXECUTION
# ============================================

print("\n" + "="*60)
print("📊 QUANTIZZAZIONI DISPONIBILI")
print("="*60)
print(f"{'#':<3} {'Nome':<10} {'Bits':<6} {'Tipo':<8} {'Qualità':<10} Descrizione")
print("-"*100)

for key, info in QUANTIZATIONS.items():
    print(f"{key:<3} {info['name']:<10} {info['bits']:<6} {info['type']:<8} {info['quality']:<10} {info['desc']}")

selected_quants = parse_quant_selection(quant_choice)
print(f"\n✅ Selezionate: {', '.join(selected_quants)}")

# Determina la cartella di output
ROOT_DIR = "/content"
if "project" in folder_structure:
    MAIN_DIR = os.path.join(ROOT_DIR, "drive/MyDrive/quantized_models")
else:
    MAIN_DIR = os.path.join(ROOT_DIR, "drive/MyDrive/quantized_models")

# Estrai il nome del modello dall'URL
if "huggingface.co/" in model_url:
    repo_path = model_url.split("huggingface.co/")[1]
else:
    repo_path = model_url

model_name = repo_path.split('/')[-1]

# Cartella di output con nome del modello
OUTPUT_FOLDER = os.path.join(MAIN_DIR, model_name)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"\n📁 Output folder: {OUTPUT_FOLDER}")
print(f"🌐 Model: {model_name}")
print(f"📦 Quantization selection: {quant_choice}")

# Download del modello
model_file = download_model()

if not model_file:
    print("\n💥 Error: Failed to download model.")
    raise SystemExit

# Esegui quantizzazioni
print("\n" + "="*60)
print("🚀 STARTING QUANTIZATION")
print("="*60)

results = {}
total_start = time.time()

for i, quant_name in enumerate(selected_quants, 1):
    if clear_output_between_quants:
        clear_output(wait=True)
        print(f"[{i}/{len(selected_quants)}] Processing {quant_name}...")
        print(f"📦 Model: {model_name}")
        print("="*60)

    success = quantize_model(model_file, quant_name, model_name, OUTPUT_FOLDER)
    results[quant_name] = success

    if i < len(selected_quants):
        time.sleep(2)

total_elapsed = time.time() - total_start

# Report finale
clear_output(wait=True)
print("\n" + "="*60)
print("📊 QUANTIZATION REPORT")
print("="*60)

successful = [q for q, r in results.items() if r]
failed = [q for q, r in results.items() if not r]

print(f"\n✅ Successful: {len(successful)}/{len(results)}")
print(f"⏱️ Total time: {total_elapsed/60:.1f} minutes")

if successful:
    print("\n📦 Generated models:")
    total_size = 0
    for quant in successful:
        file_path = os.path.join(OUTPUT_FOLDER, f"{model_name}-{quant}.gguf")
        if os.path.exists(file_path):
            size = os.path.getsize(file_path) / (1024**3)
            total_size += size
            print(f"  ✓ {model_name}-{quant}.gguf ({size:.2f} GB)")

    print(f"\n💾 Total size: {total_size:.2f} GB")
    print(f"📁 Location: {OUTPUT_FOLDER}")

if failed:
    print(f"\n⚠️ Failed: {failed}")

if successful:
    display(Markdown(f"### ✅ Done! [Go download your models from Google Drive](https://drive.google.com/drive/my-drive)\n"
                     f"### 🚀 To use a model: `ollama run {model_name}:q4_K_M`"))
else:
    print("\n❌ No models were successfully quantized.")

## *️⃣ Extras

Puoi eseguire queste celle prima di iniziare la quantizzazione.

In [ ]:
#@markdown ### 💾 Check Disk Space

import shutil

total, used, free = shutil.disk_usage("/content")
print(f"💾 Disk space in /content:")
print(f"   Free: {free / (1024**3):.2f} GB")
print(f"   Used: {used / (1024**3):.2f} GB")
print(f"   Total: {total / (1024**3):.2f} GB")

if free / (1024**3) < 10:
    print("\n⚠️ Low disk space! Consider freeing up space before proceeding.")

In [ ]:
#@markdown ### 🧹 Cleanup Temporary Files

print("🧹 Cleaning up temporary files...")

# Pulisci file Modelfile temporanei
temp_files = Path("/tmp").glob("Modelfile_*")
for f in temp_files:
    try:
        f.unlink()
        print(f"  Removed: {f.name}")
    except:
        pass

# Pulisci log di Ollama
if os.path.exists("/tmp/ollama.log"):
    os.remove("/tmp/ollama.log")
    print("  Removed: ollama.log")

print("\n✅ Cleanup completed")

# Opzionale: rimuovi modello originale per liberare spazio
if 'model_file' in locals() and model_file and os.path.exists(model_file):
    print(f"\n⚠️ Original model is at: {model_file}")
    print("To free up space, you can delete it manually if you no longer need it.")